# 🤘 Cache Me If You Can — Act IV: the sting

## The Speedrun

In Act II you built a working KV cache into GPT-2. It produces one token in about 20 ms. Now make it **as fast as you can**: same model, same card, same answers.

**How the race works.** The race feeds five short texts to your model, one token at a time. The first 128 tokens of each text go in at once (the prompt); then 128 more go in one by one. After every token, your model says which token it expects next. That is exactly the work of generating text, but because the text comes from outside, nobody can work out the answers in advance.

**The rules.**

- **Score:** the median time per token, in milliseconds. Lower is better.
- **Quality:** your model must guess the real next token about as often as the original GPT-2, at most 3 percentage points less. Inside that limit, anything goes: rewrite the code, change number formats, skip work you don't need.
- **Not timed:** loading, converting weights, preparing anything before the first token. A real server does that once, before the users arrive.
- **How you compare:** section 5 has the times the reference solution reaches.
- **Verified:** the race measures the speed and checks the quality itself, so any result can be checked by running the race again. Times depend on the card, so the result says which card it ran on.

**If the card stops responding** (a cell runs forever, or every operation fails after an error while recording): use *Restart* on the notebook's kernel, then run the cells from the top again. Your code stays in the notebook.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()                  # the folder that holds common/ and notebooks/
while ROOT != ROOT.parent and not (ROOT / "common" / "speedrun.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common import config, wait_for_warmup
wait_for_warmup()
import json, math, os, time
import torch, ttnn
from common import speedrun
from common.gpt2 import GPT2, causal_mask

device = ttnn.open_device(device_id=0, trace_region_size=config.TRACE_REGION_BYTES)
model = GPT2(device, config.local_weights(config.GPT2_MODEL_ID))
H, D_HEAD, D = model.n_heads, model.d_head, model.d_model
PRECISE = model.precise       # add up products in 32-bit: the faithful GPT-2 of Act II
print(f"GPT-2 on {device.arch()}: {model.n_layers} layers, {H} heads of size {D_HEAD}")

## 1. The starting point

This is your Act II code, written out in full: every operation of every layer is visible. The race expects two methods:

- `prefill(token_ids)`: reads the prompt, returns the predicted next token. It also starts a new text, so it must forget the previous one.
- `decode(token, position)`: reads one more token, returns the predicted next token.

Run the race once to see where you start. The first race also measures the original model's quality on the five texts, which takes a few extra seconds.

In [ ]:
def attention(q, k, v, mask):
    scores = ttnn.matmul(q, ttnn.permute(k, (0, 1, 3, 2))) * (1.0 / math.sqrt(D_HEAD))
    if mask is not None:
        scores = ttnn.add(scores, mask)
    return ttnn.matmul(ttnn.softmax(scores, dim=-1), v)


class Starter:
    def __init__(self, model, device):
        self.m, self.device = model, device

    def forward(self, new_ids):
        m, T = self.m, len(new_ids)
        x = m.embed(new_ids, start_pos=self.length)                  # embeddings: looked up on the host, then copied
        for i, L in enumerate(m.layers):
            h = ttnn.layer_norm(x, weight=L["ln1_w"], bias=L["ln1_b"], epsilon=1e-5, compute_kernel_config=PRECISE)
            qkv = ttnn.linear(h, L["w_qkv"], bias=L["b_qkv"], compute_kernel_config=PRECISE)
            split = lambda t: ttnn.permute(ttnn.reshape(t, (1, T, H, D_HEAD)), (0, 2, 1, 3))
            q, k, v = split(qkv[:, :, :D]), split(qkv[:, :, D:2 * D]), split(qkv[:, :, 2 * D:])
            if self.k[i] is not None:                                # the KV cache from Act II
                k, v = ttnn.concat([self.k[i], k], dim=2), ttnn.concat([self.v[i], v], dim=2)
            self.k[i], self.v[i] = k, v
            a = attention(q, k, v, causal_mask(self.device, n_new=T, n_total=k.shape[2]))
            a = ttnn.reshape(ttnn.permute(a, (0, 2, 1, 3)), (1, T, D))
            x = ttnn.add(x, ttnn.linear(a, L["w_o"], bias=L["b_o"], compute_kernel_config=PRECISE))
            h = ttnn.layer_norm(x, weight=L["ln2_w"], bias=L["ln2_b"], epsilon=1e-5, compute_kernel_config=PRECISE)
            h = ttnn.gelu(ttnn.linear(h, L["w_fc"], bias=L["b_fc"], compute_kernel_config=PRECISE))
            x = ttnn.add(x, ttnn.linear(h, L["w_proj"], bias=L["b_proj"], compute_kernel_config=PRECISE))
        self.length += T
        return int(m.next_token_logits(x).argmax())                  # all 50,257 scores go to the host

    def prefill(self, token_ids):
        self.k, self.v, self.length = [None] * self.m.n_layers, [None] * self.m.n_layers, 0
        return self.forward(token_ids)

    def decode(self, token, position):
        return self.forward([token])


start = speedrun.race(Starter, model, device)

## 2. Where does the time go?

GPT-2 small has 124 million parameters. Reading all of them once from memory takes well under a millisecond on this card, and the arithmetic for one token is tiny too. So why does a token take about 20 ms?

Count the operations in `Starter.forward`: roughly 26 per layer (layer norms, linears, slices, reshapes, permutes, the cache concatenations, the attention steps, the additions), times 12 layers. About **300 operations per token**, and Python sends each one to the card as a separate command. Measure what one command costs when it does almost nothing:

In [ ]:
tiny = ttnn.from_torch(torch.ones(32, 32, dtype=torch.bfloat16), layout=ttnn.TILE_LAYOUT, device=device)

def three_hundred_additions():
    x = tiny
    for _ in range(300):
        x = ttnn.add(x, tiny)
    return x

sent = speedrun.time_sent(device, three_hundred_additions)
replayed = speedrun.time_traced(device, three_hundred_additions)
print(f"300 tiny additions, each sent from Python: {sent:5.1f} ms  ({1000 * sent / 300:4.0f} us per command)")
print(f"the same 300, recorded once and replayed:  {replayed:5.1f} ms  ({1000 * replayed / 300:4.0f} us per command)")

Three hundred additions that do almost nothing take about as long as a whole token of GPT-2. The starting code spends nearly all of its time **sending commands**, not computing.

There are two ways out, and you can use both:

1. **Send fewer commands.** Many groups of operations have a single fused operation that does the same work.
2. **Record the commands once and replay them** for every token. This is **trace**, which you met in Act I. The card replays the recording on its own, with no Python in between.

## 3. Toolbox

Recipes for the operations you will need, with GPT-2's shapes. For anything else, `help(ttnn.<name>)` and the [TT-NN API reference](https://docs.tenstorrent.com/tt-metal/latest/ttnn/ttnn/api.html) list every argument. Nothing here is a complete solution; putting the pieces together is the race.

**Measure before you optimise.** `speedrun.time_sent(device, fn)` times any piece of code as it runs normally. `speedrun.time_traced(device, fn)` times it recorded and replayed: the time the card itself needs. For example, one linear layer:

In [ ]:
L0 = model.layers[0]
x1 = ttnn.from_torch(torch.randn(1, 1, D, dtype=torch.bfloat16), layout=ttnn.TILE_LAYOUT, device=device)
print(f"one linear 768 -> 2304: sent {speedrun.time_sent(device, lambda: ttnn.linear(x1, L0['w_qkv'], bias=L0['b_qkv'])):.3f} ms, "
      f"on the card {speedrun.time_traced(device, lambda: ttnn.linear(x1, L0['w_qkv'], bias=L0['b_qkv'])):.3f} ms")

**Fewer commands** (`T` is the number of tokens in the call):

```python
# split q, k, v into heads in one operation instead of 3 slices + 3 reshapes + 3 permutes
q, k, v = ttnn.experimental.nlp_create_qkv_heads(ttnn.reshape(qkv, (1, 1, T, 3 * D)), num_heads=H,
                                                 num_kv_heads=H, transpose_k_heads=False)   # each [1, H, T, 64]
# merge the heads back in one operation
a = ttnn.reshape(ttnn.experimental.nlp_concat_heads(a), (1, T, D))
# the activation inside the linear layer
h = ttnn.linear(h, w, bias=b, activation="gelu")
# all of attention in one operation (for the prompt)
a = ttnn.transformer.scaled_dot_product_attention(q, k, v, is_causal=True, scale=1 / math.sqrt(D_HEAD))
```

**Number formats and where tensors live** (Act III explained `bfloat8_b`). The starting code adds up products in 32-bit (`PRECISE`), which keeps this GPT-2 as accurate as the original. Cheaper settings are faster and less exact; the quality rule decides how far you can go:

```python
w8 = ttnn.typecast(w, ttnn.bfloat8_b)                                  # weights in bfloat8_b: half the bytes
lofi = ttnn.init_device_compute_kernel_config(device.arch(), math_fidelity=ttnn.MathFidelity.LoFi)
y = ttnn.linear(x, w8, bias=b, compute_kernel_config=lofi,             # faster, less exact multiplication
                memory_config=ttnn.L1_MEMORY_CONFIG)                   # keep the result in the cores' own memory
```

**The production KV cache.** Trace needs every shape to stay the same from token to token, so the cache can no longer grow by concatenation. Instead it is allocated once at full size, and each new key and value is written in place, at a position that is itself stored on the card. Special single-token ("decode") operations split the heads, update the cache and run attention. The cell below checks them against plain PyTorch:

In [ ]:
heads_layout, merge_layout = speedrun.decode_layouts(device)        # where these operations want their data
MAX_LEN = 1024                                                     # GPT-2's longest context
zeros = torch.zeros(1, H, MAX_LEN, D_HEAD, dtype=torch.bfloat16)
k_cache = ttnn.from_torch(zeros, layout=ttnn.TILE_LAYOUT, device=device)
v_cache = ttnn.from_torch(zeros, layout=ttnn.TILE_LAYOUT, device=device)

# pretend a 40-token prompt is already stored: fill_cache writes a whole block at the start
k_old, v_old = torch.randn(1, H, 40, D_HEAD), torch.randn(1, H, 40, D_HEAD)
ttnn.fill_cache(k_cache, ttnn.from_torch(k_old.bfloat16(), layout=ttnn.TILE_LAYOUT, device=device), 0)
ttnn.fill_cache(v_cache, ttnn.from_torch(v_old.bfloat16(), layout=ttnn.TILE_LAYOUT, device=device), 0)

# one new token at position 40; its fused q, k, v come out of the qkv linear as [1, 1, 1, 3 * D]
position = ttnn.from_torch(torch.tensor([40], dtype=torch.int32), dtype=ttnn.int32, device=device)
qkv_new = torch.randn(1, 1, 1, 3 * D)
qkv = ttnn.from_torch(qkv_new.bfloat16(), layout=ttnn.TILE_LAYOUT, device=device)

q, k, v = ttnn.experimental.nlp_create_qkv_heads_decode(qkv, num_heads=H, num_kv_heads=H, memory_config=heads_layout)
ttnn.experimental.paged_update_cache(k_cache, k, update_idxs_tensor=position)    # write in place at `position`
ttnn.experimental.paged_update_cache(v_cache, v, update_idxs_tensor=position)
a = ttnn.transformer.scaled_dot_product_attention_decode(q, k_cache, v_cache, cur_pos_tensor=position,
                                                          scale=1 / math.sqrt(D_HEAD))  # attends to 0..position
a = ttnn.experimental.nlp_concat_heads_decode(ttnn.to_memory_config(a, merge_layout), num_heads=H)
a = ttnn.reshape(a, (1, 1, 1, D), (1, 1, 32, D))                   # [1, 1, 1, 768], ready for the output linear

# the same in plain PyTorch
qn, kn, vn = qkv_new[0, 0, 0].split(D)
K = torch.cat([k_old[0], kn.view(H, 1, D_HEAD)], 1)
V = torch.cat([v_old[0], vn.view(H, 1, D_HEAD)], 1)
expected = (torch.softmax(qn.view(H, 1, D_HEAD) @ K.transpose(1, 2) / 8, -1) @ V).reshape(-1)
got = ttnn.to_torch(a).float().reshape(-1)
print(f"decode attention differs from PyTorch by {100 * ((got - expected).norm() / expected.norm()):.1f}%  (bfloat16 rounding)")

**Trace.** Three rules: shapes never change; nothing inside the recorded step copies data to or from the host; inputs sit in fixed tensors on the card that you overwrite before every replay.

```python
x_in = ttnn.from_torch(torch.zeros(1, 1, 1, D, dtype=torch.bfloat16), layout=ttnn.TILE_LAYOUT, device=device)
pos_in = ttnn.from_torch(torch.zeros(1, dtype=torch.int32), dtype=ttnn.int32, device=device)

def step():                     # uses only x_in, pos_in and tensors already on the card
    ...
    return next_token           # a tensor on the card

step()                          # run once first: this compiles every kernel
trace = ttnn.begin_trace_capture(device, cq_id=0)
out = step()                    # recorded, not run
ttnn.end_trace_capture(device, trace, cq_id=0)

# for every token: overwrite the inputs, replay, read the answer
ttnn.copy_host_to_device_tensor(ttnn.from_torch(new_x, layout=ttnn.TILE_LAYOUT), x_in)
ttnn.copy_host_to_device_tensor(ttnn.from_torch(torch.tensor([pos], dtype=torch.int32), dtype=ttnn.int32), pos_in)
ttnn.execute_trace(device, trace, cq_id=0, blocking=False)
token = int(ttnn.to_torch(out).flatten()[-1])
```

The input embedding of a token is `model.wte[token] + model.wpe[pos]`, two rows of tables kept on the host. `ttnn.argmax(logits, dim=-1)` picks the most likely token on the card. Add a `close()` method that calls `ttnn.release_trace(device, trace)`, so the race can free the recording when it is done.

**One more hint.** Once your step is traced, use `time_traced` on its pieces. At least one of them takes far longer than you would guess, and `help()` on that operation explains why.

## 4. Your move

`MyGenerator` starts as a copy of `Starter`. Change anything, run the race, repeat. Ideas roughly from easy to hard: fused operations, keeping work on the card, number formats and L1, the production cache, then trace. Measure each change; not every idea pays off.

In [ ]:
class MyGenerator(Starter):
    """Copy the methods you want to change from Starter into this class, then change them."""
    pass


mine = speedrun.race(MyGenerator, model, device)

## 5. How you compare

In the race on an n150, the starting code takes about 24 ms per token and the reference solution about 4.1 ms (2.6 ms on a Blackhole p100a). `solutions/04_the_sting.ipynb` builds it up step by step and shows what each step saved.

**Free the card for the next act.** A program that has used the card keeps it until it stops, so the last cell stops this notebook's kernel. Your outputs stay on the page; VS Code will show the kernel as stopped, which is expected.

In [ ]:
ttnn.close_device(device)
print("Act IV done.")
from common import release_card
release_card()